#Definición de la variable objetivo

La variable objetivo que elegimos es \texttt{incidencia_delictiva}, porque representa la cantidad de delitos registrados en la base de datos para combinaciones de datos como la entidad federativa, el tipo de delito, el subtipo, la modalidad y el periodo temporal.

La elección de esta variable esta basada en que es el principal indicador cuantitativo del fenómeno estudiado (analizar y comprender el comportamiento de la incidencia delictiva en México) por lo que buscar que el modelo busque estimar el número de delitos que pueden ocurrir bajo determinadas condiciones.

Ventajas:

1. \textbf{Representa el fenomeno de interés de manera directa}. Mientras que otras variables del dataset describen características o categorías de delitos, \texttt{incidencia_delictiva} mide el resultado que se desea explicar y predecir

2. \textbf{Permite identificar patrones de espaciales y temporales}. Al relacionar la incidencia delictiva con variables como la entidad federativa, el tipo de delito y la fecha, es posible detectar tendencias y diferencias regionales que pueden ser aprovechadas por el modelo.

In [1]:
import numpy as np
import pandas as pd
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.dummy import DummyRegressor
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score)
import matplotlib.pyplot as plt

#Preprocesamiento

Como el dataset ya fue sometido a un proceso de limpieza y normalización en la etapa de análisis exploratorio, para el preprocesamiento separamos la fecha en día, mes. anio para poder análizar con más facilidad la temporalidad, además, aplicamos one-hot encoding a las variables categóricas que vienen en el dataset (entidad, tipo_delito, subtipo_delito, modalidad, bien_juridico_afectado).

Debido a que desde la etápa de análisis exploratorio encotramos que la variable \texttt{incidencia_delictiva} tiene un sesgo marcado mediante np.log1p para establizar su varianza

In [2]:
RANDOM_STATE = 42

def preparar_datos(df: pd.DataFrame):
    '''
    Ajusta nombres de columnas para que sean como en el EDA
    '''
    objetivo = "incidencia_delictiva"

    df = df.copy()

    if "fecha" in df.columns:
        df["fecha"] = pd.to_datetime(df["fecha"])
        df["anio"] = df["fecha"].dt.year
        df["mes"] = df["fecha"].dt.month
        df = df.drop(columns = ["fecha"])
    
    y = np.log1p(df[objetivo])
    X = df.drop(columns=[objetivo])

    return X, y

def construir_pipeline(X):
    categoricas = X.select_dtypes(include=["object", "category"]).columns.tolist()

    numericas = [c for c in X.columns if c not in categoricas]

    preprocessor = ColumnTransformer(
        transformers=[
            (
                "cat",
                Pipeline(
                    [
                        ("imputer", SimpleImputer(strategy="most_frequent")),
                        ("encoder", OneHotEncoder(handle_unknown="ignore"),),
                    ]
                ),
                categoricas,
            ),
            (
                "num",
                Pipeline(
                    [
                        ("imputer", SimpleImputer(strategy="median")),
                    ]
                ),
                numericas,
            ),
        ]
    )

    pipeline = Pipeline(
        [
            ("preprocessor", preprocessor),
            (
                "model",
                RandomForestRegressor(
                    random_state=42,
                    n_jobs=-1
                ),
            ),
        ]
    )

    return pipeline
    

# Seleccion de variables de entrada

Las variables fueron elegidas considerando el análisis exploratorio. Se eligieron las siguientes variable:

 ## Entidad
 La entidad fue incluida porque el EDA mostró diferencias marcadas en los niveles de incidencia delictiva entre los estados del país. Factores como la densidad poblacional, actividad económica, ubicación geográfica y condiciones sociales influyen en la cantidad de delitos registrados por lo que consideramos que esta variable es importante para la predicción

 ## Tipo de delito
 Los distintos tipos de delitos presentan frecuencias muy diferentes. Delitos como el robo suelen tener incidencias mayores que otros menos comunes.

 ## Subtipo de delito
 Proporciona un nivel de detalle adicional dentro de cada categoría de delito. Incluir esta variable permite al modelo capturar patrones más específicos

 ## Modalidad
 La modalidad describe características bajo las cuales ocurren los delitos, complementa la clasificación principal de un delito y ayuda a diferenciar observaciones que pueden pertenecer al mismo tipo de delito pero con incidencias distintas

 ## Bien jurídico
 Esta variable agrupa delitos según el bien que resulta afectado. Durante el análisis exploratorio se observó que ciertos grupos concentran una mayor cantidad de registros, por lo que es una variable que puede aportar a la predicción

 ## Año, mes
 La incidencia delictiva cambia con el tiempo debido a factores económicos, sociales, demográficos y cambios en la política. Incorporar estas variables permite al modelo identificar tendencias a largo plazo.

## Entrenamiento y Ajustes

Se eligió el modelo Random Forest regresion ya que se puede capturar la relación entre las variables predictoras y la variable objetivo.

## Configuración del modelo
El modelo se inicializó con una semilla fija para garantizar la reproducibilidad de los resultados y posteriormente se construye un pipeline que une el preprocesamiento de los datos  y el modelo de aprendizaje, lo que nos permitió que todas las transformaciones se aplicaran de manera consistente.

## Ajuste de hiperparámetros
Con el objetivo de mejorar el modelo realizamos una búsqueda de hiperparámetros mediante la técnica de grid search y validación cruzada. Los hiperparámetros que usamos fueron:

* n_estimators : entre 100 y 200 (cantidad de árboles en el bosque)
* max depth : entre 10 y 20 (máxima profundidad de los árboles)
* min_samples_split : 2, 5 (número mínimo de observaciones necesarias en las que se puede dividir un nodo)
* min_samples_leaf : 1,2 (máximo número de observaciones que debe contener una hoja)

Para evaluar cada combinación de hiperparámetros usamos la validación cruzada de 5 particiones (cuatro subconjuntos para el entrenamiento y uno para la validación) , esta técnica reduce el riesgo de sobreajuste y proporciona una estimación aceptable de la generalización del modelo

## Selección del mejor modelo
La métrica utilizada durante el proceso de optimización fue R^2 ya que nos permite medir qué proporción de la variabilidad de la incidencia delictiva es explicada por el modelo.

Al finalizar la búsqueda, se selecciona automáticamente la combinación de hiperparámetros que obtuvo el mejor resultado durante la validación cruzada

In [3]:
def evaluar(nombre, y_true, y_pred):

    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)

    print(f"\n=== {nombre} ===")
    print(f"MAE  : {mae:.4f}")
    print(f"MSE  : {mse:.4f}")
    print(f"RMSE : {rmse:.4f}")
    print(f"R²   : {r2:.4f}")

    return {
        "Modelo": nombre,
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "R2": r2,
    }


def grafica_residuales(y_true, y_pred):

    residuales = y_true - y_pred

    plt.figure(figsize=(8, 5))
    plt.scatter(y_pred, residuales)
    plt.axhline(y=0)
    plt.xlabel("Predicciones")
    plt.ylabel("Residuales")
    plt.title("Gráfica de residuales")
    plt.tight_layout()
    plt.show()

In [ ]:
def main():

    DATA_PATH = "../data/INM_estatal_dic25.csv"

    df = pd.read_csv(DATA_PATH)

    X, y = preparar_datos(df)

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=RANDOM_STATE,
    )

    # Baseline
    baseline = DummyRegressor(strategy="mean")
    baseline.fit(X_train.select_dtypes(include=[np.number]), y_train)

    base_pred = baseline.predict(
        X_test.select_dtypes(include=[np.number])
    )

    resultados = []

    resultados.append(
        evaluar("Baseline", y_test, base_pred)
    )

    pipeline = construir_pipeline(X)

    param_grid = {
        "model__n_estimators": [100],
        "model__max_depth": [20, None],
        "model__min_samples_split": [2],
        "model__min_samples_leaf": [1],
    }

    grid = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        cv=5,
        scoring="r2",
        n_jobs=-1,
        verbose=3
    )

    grid.fit(X_train, y_train)

    print("\nMejores parámetros")
    print(grid.best_params_)

    best_model = grid.best_estimator_

    y_pred = best_model.predict(X_test)

    resultados.append(
        evaluar("Random Forest", y_test, y_pred)
    )

    print("\nResumen")
    print(pd.DataFrame(resultados))

    grafica_residuales(y_test, y_pred)

    joblib.dump(
        best_model,
        "random_forest_incidencia.joblib",
    )

    modelo_cargado = joblib.load(
        "random_forest_incidencia.joblib"
    )

    demo_pred = modelo_cargado.predict(
        X_test.iloc[:5]
    )

    print("\nPredicciones de prueba")
    print(demo_pred)


if __name__ == "__main__":
    main()


=== Baseline ===
MAE  : 1.7278
MSE  : 3.9435
RMSE : 1.9858
R²   : -0.0000
Fitting 5 folds for each of 2 candidates, totalling 10 fits


C:\Users\yoba7\AppData\Local\Temp\ipykernel_20448\2884849985.py:23: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categoricas = X.select_dtypes(include=["object", "category"]).columns.tolist()
